In [35]:
import pandas as pd
import numpy as np

In [36]:
d1 = {'Social_media_followers':[1000000, np.nan, 2000000, 1310000, 1700000, np.nan, 4100000, 1600000, 2200000, 1000000],
'Sold_out':[1,0,0,1,0,0,0,1,0,1]}

In [37]:
df1 = pd.DataFrame(data=d1)

In [38]:
df1

,Social_media_followers,Sold_out
0,1000000.0,1
1,NaN,0
2,2000000.0,0
3,1310000.0,1
4,1700000.0,0
5,NaN,0
6,4100000.0,0
7,1600000.0,1
8,2200000.0,0
9,1000000.0,1


In [39]:
X1 = df1[['Social_media_followers']]

In [40]:
y1 = df1[['Sold_out']]

In [41]:
from sklearn.model_selection import train_test_split

In [42]:
X1_train, X1_test, y1_train, y1_test = train_test_split(X1,y1,test_size=0.3,random_state=19)


In [43]:
X1_train.shape

(7, 1)

In [44]:
X1_test.shape

(3, 1)

In [45]:
from sklearn.impute import SimpleImputer

In [46]:
# How to handle missing vars
imputer = SimpleImputer(strategy='mean')

In [47]:
from sklearn.linear_model import LogisticRegression

In [48]:
lr = LogisticRegression()

# make_pipeline Example

In [49]:
# make_pipeline example
from sklearn.pipeline import make_pipeline


In [50]:
pipe1 = make_pipeline(imputer, lr)

In [51]:
pipe1.fit(X1_train, y1_train)

c:\Users\jpadd\miniconda3\Lib\site-packages\sklearn\utils\validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Pipeline(steps=[('simpleimputer', SimpleImputer()),
                ('logisticregression', LogisticRegression())])

In [52]:
pipe1.score(X1_train, y1_train)

1.0

In [53]:
pipe1.score(X1_test, y1_test)

0.6666666666666666

In [54]:
pipe1.named_steps.simpleimputer.statistics_

array([2051666.66666667])

In [55]:
pipe1.named_steps.logisticregression.coef_

array([[-9.72872687e-05]])

# Simple Pipeline Example

In [56]:
d2 = {'Genre':['Rock', 'Metal', 'Bluegrass', 'Rock', np.nan, 'Rock', 'Rock', np.nan, 'Bluegrass', 'Rock'],
'Social_media_followers':[1000000, np.nan, 2000000, 1310000, 1700000, np.nan, 4100000, 1600000, 2200000, 1000000],
'Sold_out':[1,0,0,1,0,0,0,1,0,1]}

In [57]:
df = pd.DataFrame(data=d2)
df.head()

,Genre,Social_media_followers,Sold_out
0,Rock,1000000.0,1
1,Metal,NaN,0
2,Bluegrass,2000000.0,0
3,Rock,1310000.0,1
4,NaN,1700000.0,0


In [58]:
#Ignore Sold_out
X = df.iloc[:, 0:2]
X.head()

,Genre,Social_media_followers
0,Rock,1000000.0
1,Metal,NaN
2,Bluegrass,2000000.0
3,Rock,1310000.0
4,NaN,1700000.0


In [59]:
y = df.iloc[:, -1]
y.head()

0    1
1    0
2    0
3    1
4    0
Name: Sold_out, dtype: int64

In [60]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=75, stratify=y, test_size=0.3)

In [61]:
num_cols = ['Social_media_followers']
cat_cols = ['Genre']

In [62]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler # for numerical
from sklearn.preprocessing import OneHotEncoder # for categorical

In [63]:
num_pipeline = Pipeline(steps = [
    ('impute', SimpleImputer(strategy='mean')),
    ('scale', StandardScaler())
]
)

In [64]:
cat_pipeline = Pipeline(steps = [
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('one-hot-enc', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
]
)

In [65]:
from sklearn.compose import ColumnTransformer

In [66]:
col_transformer = ColumnTransformer(transformers = [
    ('num_pipeline', num_pipeline, num_cols),
    ('cat_pipeline', cat_pipeline, cat_cols),
],
remainder = 'drop',
n_jobs = -1
)

In [67]:
from sklearn.tree import DecisionTreeClassifier

dtc = DecisionTreeClassifier()

In [68]:
pipefinal = make_pipeline(col_transformer, dtc)
pipefinal.fit(X_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(n_jobs=-1,
                                   transformers=[('num_pipeline',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer()),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  ['Social_media_followers']),
                                                 ('cat_pipeline',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('one-hot-enc',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Genre'])])),
                ('decisiontreeclassifier', DecisionTreeClassifier())])

In [69]:
pipefinal.score(X_test, y_test)

1.0

# How to Save Pipeline Using JobLib

In [70]:
import joblib

In [76]:
joblib.dump(pipefinal, "pipe.joblib")


['pipe.joblib']

In [ ]:
# To load model
pipefinal2 = joblib.load("./pipe.joblib")